# Projet Moneyball : Modernisation du système de données

# Motivation et Méthodologie

### Pourquoi ce projet ?
J'entreprends ce projet avec l'objectif d'approfondir ma compréhension de l'analyse de données et de démontrer ma maîtrise du **SQL**. En utilisant la base de données *European Soccer*, je souhaite proposer une introduction complète à l'analyse de données via SQL — un outil indispensable pour tout Data Scientist. 

La logique et les principes du SQL sont étroitement liés à d'autres outils populaires comme **Pandas** ou **Excel**, ce qui en fait une compétence intuitive et essentielle pour quiconque travaille avec la donnée. Ma démarche ici illustre ma capacité d'**autonomie** et mon **adaptation** face à des jeux de données complexes.

---

### Ce qu'il faut comprendre
Voici les étapes structurantes que je suivrai pour extraire des informations pertinentes et construire mes rapports :

* **SELECT :** Cette étape pose les bases en sélectionnant les colonnes ou attributs spécifiques à afficher.
* **FROM :** Définit la source de récupération des données en spécifiant les tables et en préparant l'exploration.
* **JOIN :** Permet d'obtenir des insights complets en fusionnant des tables distinctes via des clés partagées (**Primary & Foreign Keys**).

* **WHERE :** Filtre les données de manière sélective pour ne conserver que les lignes répondant à des conditions précises.
* **GROUP BY :** Élève l'analyse à un niveau supérieur en regroupant les données selon des dimensions choisies pour identifier des tendances.
* **HAVING :** Ajoute une couche de filtrage supplémentaire après le regroupement, en se concentrant sur les conditions agrégées.
* **ORDER BY :** Orchestre l'arrangement des lignes de sortie, améliorant ainsi la lisibilité et la compréhension des résultats.
* **LIMIT :** Conclut le processus en limitant le nombre de lignes affichées pour garantir un rapport concis et pertinent.

## 1. La découverte

En arrivant au club, ma première mission est de comprendre comment les données sont stockées. J'interroge le dictionnaire de données (sqlite_master) pour cartographier l'architecture.

In [1]:
#Imports
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect('database.sqlite')
tables = pd.read_sql("SELECT * FROM sqlite_master WHERE type='table';", conn)

tables

,type,name,tbl_name,rootpage,sql
0,table,sqlite_sequence,sqlite_sequence,4,"CREATE TABLE sqlite_sequence(name,seq)"
1,table,Player_Attributes,Player_Attributes,11,"CREATE TABLE ""Player_Attributes"" (\n\t`id`\tIN..."
2,table,Player,Player,14,CREATE TABLE `Player` (\n\t`id`\tINTEGER PRIMA...
3,table,Match,Match,18,CREATE TABLE `Match` (\n\t`id`\tINTEGER PRIMAR...
4,table,League,League,24,CREATE TABLE `League` (\n\t`id`\tINTEGER PRIMA...
5,table,Country,Country,26,CREATE TABLE `Country` (\n\t`id`\tINTEGER PRIM...
6,table,Team,Team,29,"CREATE TABLE ""Team"" (\n\t`id`\tINTEGER PRIMARY..."
7,table,Team_Attributes,Team_Attributes,2,CREATE TABLE `Team_Attributes` (\n\t`id`\tINTE...


## 1. Fondations et structure 

In [3]:
# SELECT
countries = pd.read_sql("SELECT * FROM Country;", conn)

countries

,id,name
0,1,Belgium
1,1729,England
2,4769,France
3,7809,Germany
4,10257,Italy
5,13274,Netherlands
6,15722,Poland
7,17642,Portugal
8,19694,Scotland
9,21518,Spain


In [4]:
# UPDATE
cursor = conn.cursor()
cursor.execute(f"""
               UPDATE Country 
               SET name = 'Angleterre' 
               WHERE id = 1729 and name = 'England';
               """)
conn.commit()

# vérification de la mise à jour
updated_country = pd.read_sql("SELECT * FROM Country;", conn)

updated_country

,id,name
0,1,Belgium
1,1729,Angleterre
2,4769,France
3,7809,Germany
4,10257,Italy
5,13274,Netherlands
6,15722,Poland
7,17642,Portugal
8,19694,Scotland
9,21518,Spain


#### Le club ouvre une nouvelle cellule de recrutement. Je dois donc ajouter ce pays à notre périmètre d'analyse.

In [5]:
# insertion d'un nouveau pays
cursor.execute(f"""
               INSERT INTO Country (name) 
               VALUES ('Guinea');
               """)
conn.commit()
# vérification de l'insertion
countries_after_insertion = pd.read_sql("SELECT * FROM Country;", conn)

countries_after_insertion

,id,name
0,1,Belgium
1,1729,Angleterre
2,4769,France
3,7809,Germany
4,10257,Italy
5,13274,Netherlands
6,15722,Poland
7,17642,Portugal
8,19694,Scotland
9,21518,Spain


### Suite à un changement de stratégie budgétaire, le club décide de fermer ses activités dans ce pays pour se concentrer sur le marché britannique.

In [6]:
# suppression d'un pays
cursor.execute(f"""
               DELETE FROM Country 
               WHERE name = 'Guinea';
               """)
conn.commit()
# vérification de la suppression
countries_after_deletion = pd.read_sql("SELECT * FROM Country;", conn)

countries_after_deletion

,id,name
0,1,Belgium
1,1729,Angleterre
2,4769,France
3,7809,Germany
4,10257,Italy
5,13274,Netherlands
6,15722,Poland
7,17642,Portugal
8,19694,Scotland
9,21518,Spain


## 2. Interrogation et Logique de Filtrage

#### La direction demande une liste des championnats disponibles et une première sélection de joueurs de grand gabarit pour le jeu aérien.

In [7]:
# Utilisation de DISTINCT pour l'inventaire des ligues
leagues = pd.read_sql("""
                      SELECT DISTINCT name 
                      AS nom_championnat 
                      FROM League;""", 
                      conn)

leagues

,nom_championnat
0,Belgium Jupiler League
1,England Premier League
2,France Ligue 1
3,Germany 1. Bundesliga
4,Italy Serie A
5,Netherlands Eredivisie
6,Poland Ekstraklasa
7,Portugal Liga ZON Sagres
8,Scotland Premier League
9,Spain LIGA BBVA


In [14]:
# filtrage avec WHERE pour les joueurs de grand gabarit
tall_players = pd.read_sql("""
                           SELECT * 
                           FROM Player 
                           WHERE height >= 190
                           AND weight > 84
                           AND (birthday BETWEEN '1990-01-01' AND '2000-12-31')
                           ORDER BY height DESC, weight DESC
                           LIMIT 10
                           ;""", 
                           conn)
tall_players

,id,player_api_id,player_name,player_fifa_api_id,birthday,height,weight
0,10574,543021,Vanja Milinkovic-Savic,224836,1997-02-20 00:00:00,203.20,203
1,5957,150297,Lacina Traore,199074,1990-05-20 00:00:00,203.20,192
2,3274,601304,Fejsal Mulic,226114,1994-10-03 00:00:00,203.20,185
3,35,564712,Abdoul Ba,225050,1994-02-08 00:00:00,200.66,212
4,2106,188058,Daniel Burn,198032,1992-05-09 00:00:00,200.66,192
5,5873,361334,Konrad Jalocha,211621,1991-05-09 00:00:00,200.66,187
6,5976,210822,Lars Unnerstall,199833,1990-07-20 00:00:00,198.12,220
7,4664,205893,Jannik Vestergaard,202849,1992-08-03 00:00:00,198.12,216
8,5977,243937,Lars Veldwijk,208153,1991-08-21 00:00:00,198.12,209
9,6991,232143,Martin Polacek,229662,1990-04-02 00:00:00,198.12,209


## 3. Analyse et Fonctions d'Agrégation

#### Nous devons identifier quelles ligues sont les plus spectaculaires pour orienter les parieurs.

In [9]:
# Calculs statistiques et filtrage avec HAVING
leagues_stats = pd.read_sql("""
                      SELECT league_id,
                             COUNT(*) AS nb_matchs,
                             MIN(home_team_goal + away_team_goal) AS min_buts,
                             MAX(home_team_goal + away_team_goal) AS max_buts,
                             ROUND(AVG(home_team_goal + away_team_goal), 2) AS moyenne_buts
                      FROM Match
                    GROUP BY league_id
                    HAVING nb_matchs > 500
                    ORDER BY moyenne_buts DESC;""",
                      conn)
leagues_stats

,league_id,nb_matchs,min_buts,max_buts,moyenne_buts
0,13274,2448,0,10,3.08
1,24558,1422,0,9,2.93
2,7809,2448,0,11,2.90
3,1,1728,0,9,2.80
4,21518,3040,0,12,2.77
5,1729,3040,0,10,2.71
6,19694,1824,0,12,2.63
7,10257,3017,0,9,2.62
8,17642,2052,0,9,2.53
9,4769,3040,0,10,2.44


## 4. Utilisation des Jointures

#### Croisons les données pour mettre des noms sur les IDs de ligues et d'équipes.

In [11]:
# jointure multiple
resultat_join = pd.read_sql("""
                    SELECT L.name AS ligue, 
                        T.team_long_name AS equipe, 
                        M.season, 
                        M.home_team_goal AS buts_domicile
                    FROM Match M
                    JOIN League L ON M.league_id = L.id
                    JOIN Team T ON M.home_team_api_id = T.team_api_id
                    WHERE L.name LIKE '%Spain%'
                    LIMIT 10;""", 
                    conn)
resultat_join

,ligue,equipe,season,buts_domicile
0,Spain LIGA BBVA,Valencia CF,2008/2009,3
1,Spain LIGA BBVA,CA Osasuna,2008/2009,1
2,Spain LIGA BBVA,RC Deportivo de La Coruña,2008/2009,2
3,Spain LIGA BBVA,CD Numancia,2008/2009,1
4,Spain LIGA BBVA,Racing Santander,2008/2009,1
5,Spain LIGA BBVA,Real Sporting de Gijón,2008/2009,1
6,Spain LIGA BBVA,Real Betis Balompié,2008/2009,0
7,Spain LIGA BBVA,RCD Espanyol,2008/2009,1
8,Spain LIGA BBVA,Athletic Club de Bilbao,2008/2009,1
9,Spain LIGA BBVA,Atlético Madrid,2008/2009,4


## Les Sous-requêtes

#### Trouvons les joueurs qui ont un potentiel strictement supérieur à la moyenne mondiale.

In [12]:
# utilisation de sous-requête imbriquée
joueurs = pd.read_sql("""
        SELECT player_api_id, overall_rating, potential
        FROM Player_Attributes
        WHERE potential > (SELECT AVG(potential) FROM Player_Attributes)
        AND date LIKE '2015%'
        LIMIT 10;""", conn)

joueurs

,player_api_id,overall_rating,potential
0,155782,73,75
1,155782,73,75
2,155782,73,77
3,155782,74,78
4,155782,73,77
5,155782,71,75
6,27316,77,77
7,27316,77,77
8,27316,77,77
9,27316,77,78


## 6. Fonctions de Transformation

#### Préparation du catalogue final pour le coach avec un formatage propre et une segmentation par profil.

In [13]:
catalogue_final = pd.read_sql("""
                    SELECT 
                        'JOUEUR : ' || UPPER(player_name) AS etiquette_joueur,
                        height AS taille_cm,
                        REPLACE(birthday, '-', '/') AS date_naissance_formatee,
                        CASE 
                            WHEN height >= 190 THEN 'Profil Pivot'
                        WHEN height BETWEEN 175 AND 189 THEN 'Profil Polyvalent'
                        ELSE 'Profil Agilité'
                    END AS segment_physique
                FROM Player
                WHERE player_name LIKE 'A%'
                LIMIT 20;""", conn)

catalogue_final

,etiquette_joueur,taille_cm,date_naissance_formatee,segment_physique
0,JOUEUR : AARON APPINDANGOYE,182.88,1992/02/29 00:00:00,Profil Polyvalent
1,JOUEUR : AARON CRESSWELL,170.18,1989/12/15 00:00:00,Profil Agilité
2,JOUEUR : AARON DORAN,170.18,1991/05/13 00:00:00,Profil Agilité
3,JOUEUR : AARON GALINDO,182.88,1982/05/08 00:00:00,Profil Polyvalent
4,JOUEUR : AARON HUGHES,182.88,1979/11/08 00:00:00,Profil Polyvalent
5,JOUEUR : AARON HUNT,182.88,1986/09/04 00:00:00,Profil Polyvalent
6,JOUEUR : AARON KUHL,172.72,1996/01/30 00:00:00,Profil Agilité
7,JOUEUR : AARON LENNON,165.10,1987/04/16 00:00:00,Profil Agilité
8,JOUEUR : AARON LENNOX,190.50,1993/02/19 00:00:00,Profil Pivot
9,JOUEUR : AARON MEIJERS,175.26,1987/10/28 00:00:00,Profil Polyvalent


# Conclusion de l'Analyse de Données SQL
**Par : Ibrahima Sory DIALLO**

---

### Pour aller plus loin
N'hésitez pas à explorer davantage le schéma de la base de données, à expérimenter différentes optimisations de requêtes ou à proposer des améliorations pour accroître l'efficacité et la performance du système. 

L'analyse de données est un processus itératif : chaque nouvelle question peut révéler une pépite pour notre stratégie de recrutement !

---

### 🤝 Restons en contact
Si vous avez des questions sur cette analyse ou si vous souhaitez collaborer sur des projets similaires, vous pouvez me joindre via :

* [**LinkedIn**](https://www.linkedin.com/in/ibrahima-sory-diallo-isd/)
* 📧 [**Envoyer un Email**](mailto:isddiallopro@gmail.com)

---
*Projet réalisé dans le cadre d'un projet d'école (2026).*